In [1]:
import numpy as np
import joblib

def export_svc_to_npz(best_svc, out_npz_path, img_size=28):
    """
    Exports an sklearn.svm.SVC (rbf) into a NumPy-friendly .npz.
    Works for multiclass One-vs-One.
    """
    svc = best_svc.named_steps["svc"] if hasattr(best_svc, "named_steps") else best_svc

    if svc.kernel != "rbf":
        raise ValueError("This exporter currently supports only RBF kernel SVC.")

    data = {
        "kernel": np.array(["rbf"]),
        "gamma": np.array([svc._gamma], dtype=np.float64),  # resolved gamma value
        "support_vectors": svc.support_vectors_.astype(np.float64),
        "dual_coef": svc.dual_coef_.astype(np.float64),     # shape: (n_classes-1, n_SV)
        "intercept": svc.intercept_.astype(np.float64),     # shape: (n_pairs,)
        "n_support": svc.n_support_.astype(np.int64),       # per class
        "classes": svc.classes_.astype(np.int64),
        "probA": (svc.probA_.astype(np.float64) if hasattr(svc, "probA_") else np.array([])),
        "probB": (svc.probB_.astype(np.float64) if hasattr(svc, "probB_") else np.array([])),
        "img_size": np.array([img_size], dtype=np.int64),
    }
    np.savez(out_npz_path, **data)
    print(f"[INFO] Exported to: {out_npz_path}")


In [ ]:
import numpy as np

def export_svc_to_npz(best_pipeline_or_svc, out_npz_path: str):
    svc = best_pipeline_or_svc.named_steps["svc"] if hasattr(best_pipeline_or_svc, "named_steps") else best_pipeline_or_svc

    data = {
        "gamma": np.array([svc._gamma], dtype=np.float64),
        "support_vectors": svc.support_vectors_.astype(np.float64),
        "dual_coef": svc.dual_coef_.astype(np.float64),
        "intercept": svc.intercept_.astype(np.float64),
        "n_support": svc.n_support_.astype(np.int64),   # <-- THIS ONE
        "classes": svc.classes_.astype(np.int64),
    }
    np.savez(out_npz_path, **data)
    print(f"[INFO] Saved: {out_npz_path}")

In [3]:
BUNDLE_PATH = "svm_ocr_arialv2.pkl"  # el .joblib que guardaste al entrenar
bundle = joblib.load(BUNDLE_PATH)
export_svc_to_npz(bundle, "svc_digit_rbf.npz", img_size=28)


[INFO] Exported to: svc_digit_rbf.npz


In [8]:
# ======== CONFIG ========
import os
import glob
import cv2
import numpy as np
DATA_DIR = r"..\..\Data\dataset\text_arial"  # carpeta con subdirs 0..9
IMG_SIZE = 28            # imágenes 28x28

# ======== UTILS ========
def ensure_binary_01(img):
    """Return image in {0,1} with uint8 input possible (0..255)."""
    if img.dtype != np.uint8:
        img = img.astype(np.uint8)
    # Otsu binarization for safety
    _, bin_img = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    bin01 = (bin_img > 0).astype(np.float32)  # 0/1 float
    return bin01

def maybe_invert(bin01):
    """
    Ensure foreground is 1 and background is 0.
    If the image has mostly ones, assume it is inverted (white bg / black digit) and invert.
    """
    mean_val = bin01.mean()
    # If background dominates: mean should be low; if mean > 0.5, likely inverted
    if mean_val > 0.5:
        return 1.0 - bin01
    return bin01
def load_dataset_from_folders(data_dir, img_size=28):
    """
    Expects directory structure:
      data_dir/
        0/*.png
        1/*.png
        ...
        9/*.png
    Returns X (n_samples, n_features), y (n_samples,)
    """
    X, y = [], []
    classes = [str(i) for i in range(10)]
    total = 0

    for cls in classes:
        cls_dir = os.path.join(data_dir, cls)
        if not os.path.isdir(cls_dir):
            print(f"[WARN] Missing class dir: {cls_dir}")
            continue
        paths = glob.glob(os.path.join(cls_dir, "*.png"))
        for p in paths:
            img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"[WARN] Skipping unreadable: {p}")
                continue
            # Resize just in case
            if img.shape != (img_size, img_size):
                img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_NEAREST)

            bin01 = ensure_binary_01(img)
            bin01 = maybe_invert(bin01)  # ensure foreground=1
            # Raw pixels flattened in [0,1]
            feat = bin01.flatten()

            X.append(feat)
            y.append(int(cls))
            total += 1

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)
    print(f"[INFO] Loaded {total} samples. X shape: {X.shape}, y shape: {y.shape}")
    return X, y

In [9]:
print("[INFO] Loading dataset...")
X, y = load_dataset_from_folders(DATA_DIR, IMG_SIZE)

[INFO] Loading dataset...
[INFO] Loaded 10000 samples. X shape: (10000, 784), y shape: (10000,)


In [11]:
from OCR_numpy_predictor import NumpySVC

# sklearn predictions
sk_pred = bundle.predict(X)

# numpy predictions
clf_np = NumpySVC("svc_digit_rbf.npz")
np_pred = clf_np.predict(X)

same = (sk_pred == np_pred)
print("[CHECK] Same predictions:", same.mean(), f"({same.sum()}/{same.size})")

# If mismatch, show a few examples
idx = np.where(~same)[0]
print("[CHECK] Mismatches:", idx.size)
if idx.size > 0:
    print("First mismatches idx:", idx[:10])
    i = idx[0]
    print("sk:", sk_pred[i], "np:", np_pred[i])


[CHECK] Same predictions: 1.0 (10000/10000)
[CHECK] Mismatches: 0
